# Corporate Default Loss Analysis
### Predicting Corporate Bankruptcy in the American Stock Market Using Machine Learning

**Author:** Tianfan (Jaycee) Wu  
**Course:** AIML-500 — Machine Learning Fundamentals  
**Institution:** Indiana Wesleyan University  
**Professor:** Antonio MK Singleton  
**Date:** March 2026

---

## Project Overview

This notebook develops an end-to-end ML pipeline for corporate default prediction and loss analysis using the **American Bankruptcy Dataset** (Lombardo et al., 2022) — 78,682 firm-year observations from 8,971 public companies on NYSE and NASDAQ (1999–2018).

The project goes beyond binary classification by integrating an **Expected Loss (EL = PD × LGD × EAD)** framework, bridging ML prediction with financial risk quantification.

**Key Techniques:** Gradient Boosting, Random Forest, Logistic Regression, class-balanced weighting, threshold optimization, feature importance analysis.


---
## 0. Setup & Data Loading


In [5]:
import importlib
import subprocess
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    classification_report, confusion_matrix, roc_curve, precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
ACCENT = '#1B4F72'
COLORS = ['#1B4F72', '#E74C3C', '#27AE60', '#F39C12', '#8E44AD']
FIGSIZE = (13, 6)

print('Libraries loaded successfully.')


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Load Dataset
df = pd.read_csv('american_bankruptcy_dataset.csv')

# Encode target: 'failed' -> 1, 'alive' -> 0
df['target'] = (df['status_label'] == 'failed').astype(int)

# Define temporal splits
df['split'] = 'train'
df.loc[df['fyear'].between(2012, 2014), 'split'] = 'val'
df.loc[df['fyear'] >= 2015, 'split'] = 'test'

# Feature columns
FEAT = [f'X{i}' for i in range(1, 19)]

# Industry mapping (SIC Division codes)
div_map = {
    'A': 'Agri/Forest', 'B': 'Mining', 'C': 'Construction',
    'D': 'Manufacturing', 'E': 'Transport/Util', 'F': 'Wholesale',
    'G': 'Retail', 'H': 'Finance/RE', 'I': 'Services', 'J': 'Public Admin'
}
df['industry'] = df['Division'].map(div_map)

print(f'Shape: {df.shape}')
print(f'Companies: {df["company_name"].nunique():,}')
print(f'Years: {int(df["fyear"].min())} - {int(df["fyear"].max())}')
print(f'Failed: {df["target"].sum():,} / {len(df):,} ({df["target"].mean():.2%})')
print(f'\nSplit sizes:')
print(f'  Train (1999-2011): {(df["split"]=="train").sum():,}')
print(f'  Val   (2012-2014): {(df["split"]=="val").sum():,}')
print(f'  Test  (2015-2018): {(df["split"]=="test").sum():,}')


---
## 1. Exploratory Data Analysis (EDA)
### 1.1 Class Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

# Overall class distribution
counts = df['target'].value_counts().sort_index()
bars = axes[0].bar(['Alive (0)', 'Failed (1)'], counts.values,
                    color=[COLORS[0], COLORS[1]], edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 500,
                 f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=11)

# Failure rate by split
split_rates = df.groupby('split')['target'].mean().reindex(['train', 'val', 'test'])
bars2 = axes[1].bar(split_rates.index, split_rates.values * 100,
                     color=COLORS[:3], edgecolor='white', width=0.5)
axes[1].set_title('Failure Rate by Split', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Failure Rate (%)')
for bar, v in zip(bars2, split_rates.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, v*100 + 0.15,
                 f'{v*100:.2f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('fig_01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: Training period has ~8% failure rate vs. ~2.3% in test — the declining trend matters.')


### 1.2 Temporal Default Rate Trends


In [ ]:
yearly = df.groupby('fyear').agg(
    rate=('target', 'mean'), total=('target', 'count'), defaults=('target', 'sum')
).reset_index()

fig, ax1 = plt.subplots(figsize=FIGSIZE)
ax2 = ax1.twinx()

ax1.bar(yearly['fyear'], yearly['defaults'], color=COLORS[1], alpha=0.45,
        label='Default Count', width=0.8)
ax2.plot(yearly['fyear'], yearly['rate'] * 100, color=ACCENT, linewidth=2.5,
         marker='o', markersize=6, label='Default Rate (%)')

ax1.axvspan(2007, 2009, alpha=0.08, color='red', label='GFC Period')
ax1.axvline(2012, color='gray', ls='--', alpha=0.5, label='Val split')
ax1.axvline(2015, color='gray', ls=':', alpha=0.5, label='Test split')

ax1.set_xlabel('Fiscal Year', fontsize=12)
ax1.set_ylabel('Number of Defaults', fontsize=12, color=COLORS[1])
ax2.set_ylabel('Default Rate (%)', fontsize=12, color=ACCENT)
ax1.set_title('Corporate Default Rates Over Time (1999–2018)', fontsize=14, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=10)

plt.tight_layout()
plt.savefig('fig_02_temporal_defaults.png', dpi=150, bbox_inches='tight')
plt.show()


### 1.3 Default Rate by Industry


In [ ]:
ind_stats = df.groupby('industry').agg(
    rate=('target', 'mean'), count=('target', 'count')
).sort_values('rate', ascending=True)
ind_stats = ind_stats[ind_stats['count'] >= 50]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(ind_stats.index, ind_stats['rate'] * 100, color=ACCENT, edgecolor='white')
for bar, (_, row) in zip(bars, ind_stats.iterrows()):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{row["rate"]*100:.1f}% (n={int(row["count"]):,})', va='center', fontsize=10)
ax.set_xlabel('Failure Rate (%)', fontsize=12)
ax.set_title('Corporate Default Rate by Industry (SIC Division)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_03_industry_defaults.png', dpi=150, bbox_inches='tight')
plt.show()
print('Mining (18.1%) and Retail (10.3%) have the highest failure rates.')


### 1.4 Feature Distributions — Failed vs. Alive


In [ ]:
mean_diff = df.groupby('target')[FEAT].mean().diff().iloc[-1].abs().sort_values(ascending=False)
top8 = mean_diff.head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for i, feat in enumerate(top8):
    ax = axes[i // 4, i % 4]
    for label, color, name in zip([0, 1], [COLORS[0], COLORS[1]], ['Alive', 'Failed']):
        subset = df[df['target'] == label][feat].dropna()
        q01, q99 = subset.quantile(0.01), subset.quantile(0.99)
        subset = subset.clip(q01, q99)
        ax.hist(subset, bins=50, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Failed vs. Alive (Top 8 by Mean Difference)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_04_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


### 1.5 Correlation Analysis


In [ ]:
corr_target = df[FEAT + ['target']].corr()['target'].drop('target').abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

corr_target.plot(kind='barh', ax=axes[0], color=ACCENT)
axes[0].set_title('Features by |Correlation| with Default', fontsize=13, fontweight='bold')
axes[0].set_xlabel('|Pearson Correlation|')
axes[0].invert_yaxis()

top15 = corr_target.head(15).index.tolist()
corr_matrix = df[top15].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0, ax=axes[1],
            square=True, linewidths=0.5, annot=True, annot_kws={'size': 7})
axes[1].set_title('Inter-Feature Correlation (Top 15)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_05_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 features correlated with default:')
for feat, corr in corr_target.head(10).items():
    print(f'  {feat}: {corr:.4f}')


---
## 2. Preprocessing


In [ ]:
# Temporal splits
X_train = df[df['split'] == 'train'][FEAT].copy()
y_train = df[df['split'] == 'train']['target'].copy()
X_val   = df[df['split'] == 'val'][FEAT].copy()
y_val   = df[df['split'] == 'val']['target'].copy()
X_test  = df[df['split'] == 'test'][FEAT].copy()
y_test  = df[df['split'] == 'test']['target'].copy()

# Handle infinities, fill NaN with train median
for X in [X_train, X_val, X_test]:
    X.replace([np.inf, -np.inf], np.nan, inplace=True)

train_median = X_train.median()
X_train.fillna(train_median, inplace=True)
X_val.fillna(train_median, inplace=True)
X_test.fillna(train_median, inplace=True)

# Scale for Logistic Regression
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=FEAT, index=X_train.index)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val), columns=FEAT, index=X_val.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=FEAT, index=X_test.index)

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'Failure rates — Train: {y_train.mean():.4f} | Val: {y_val.mean():.4f} | Test: {y_test.mean():.4f}')


---
## 3. Probability of Default (PD) Modeling


In [ ]:
models = {
    'Logistic Regression': (
        LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced', C=0.1),
        True  # use scaled data
    ),
    'Random Forest': (
        RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=15,
                               class_weight='balanced', random_state=42, n_jobs=-1),
        False
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                                   subsample=0.8, min_samples_leaf=20, random_state=42),
        False
    )
}

results = {}

for name, (model, use_scaled) in models.items():
    print(f'\n{"="*60}')
    print(f'Training: {name}')
    print(f'{"="*60}')

    Xtr, Xva, Xte = (X_train_scaled, X_val_scaled, X_test_scaled) if use_scaled else (X_train, X_val, X_test)
    model.fit(Xtr, y_train)

    y_val_prob  = model.predict_proba(Xva)[:, 1]
    y_test_prob = model.predict_proba(Xte)[:, 1]

    # Threshold tuning on validation set
    best_f1, best_thresh = 0, 0.5
    for thresh in np.arange(0.02, 0.60, 0.005):
        preds = (y_val_prob >= thresh).astype(int)
        f1 = f1_score(y_val, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh

    y_test_pred = (y_test_prob >= best_thresh).astype(int)

    auc_roc = roc_auc_score(y_test, y_test_prob)
    auc_pr  = average_precision_score(y_test, y_test_prob)
    f1_val  = f1_score(y_test, y_test_pred, zero_division=0)

    results[name] = {
        'model': model, 'prob': y_test_prob, 'pred': y_test_pred,
        'thresh': best_thresh, 'AUC-ROC': auc_roc, 'AUC-PR': auc_pr, 'F1': f1_val
    }

    print(f'Optimal Threshold: {best_thresh:.3f}')
    print(f'AUC-ROC: {auc_roc:.4f} | AUC-PR: {auc_pr:.4f} | F1: {f1_val:.4f}')
    print(f'\nClassification Report (Test Set):')
    print(classification_report(y_test, y_test_pred, target_names=['Alive', 'Failed']))


### 3.1 Model Comparison


In [ ]:
# Comparison table
comp = pd.DataFrame({
    name: {'AUC-ROC': r['AUC-ROC'], 'AUC-PR': r['AUC-PR'], 'F1-Score': r['F1'], 'Threshold': r['thresh']}
    for name, r in results.items()
}).T.round(4)

print('Model Comparison (Test Set):')
comp


In [ ]:
# ROC and Precision-Recall Curves
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for i, (name, r) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, r['prob'])
    axes[0].plot(fpr, tpr, color=COLORS[i], linewidth=2.5,
                 label=f'{name} (AUC={r["AUC-ROC"]:.3f})')

    prec, rec, _ = precision_recall_curve(y_test, r['prob'])
    axes[1].plot(rec, prec, color=COLORS[i], linewidth=2.5,
                 label=f'{name} (AP={r["AUC-PR"]:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Curves', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)

axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Curves', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig('fig_06_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (name, r) in enumerate(results.items()):
    cm = confusion_matrix(y_test, r['pred'])
    sns.heatmap(cm, annot=True, fmt=',', cmap='Blues', ax=axes[i],
                xticklabels=['Alive', 'Failed'], yticklabels=['Alive', 'Failed'])
    axes[i].set_title(f'{name}\n(threshold={r["thresh"]:.3f})', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Actual'); axes[i].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices — Test Set (2015–2018)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_07_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 4. Loss Analysis Layer

### 4.1 Expected Loss Framework

The Expected Loss (EL) framework is the industry standard for credit risk quantification under Basel II/III:

$$\text{Expected Loss} = PD \times LGD \times EAD$$

Where:
- **PD** (Probability of Default): Model-predicted default probability per firm
- **LGD** (Loss Given Default): Fraction of exposure lost if default occurs (Basel standard: 45% for senior unsecured)
- **EAD** (Exposure at Default): Outstanding exposure at default ($100M assumed per firm)


In [ ]:
# Use best-performing model (by AUC-ROC)
best_name = max(results, key=lambda k: results[k]['AUC-ROC'])
best_result = results[best_name]
pd_predictions = best_result['prob']
EAD = 100_000_000  # $100M per firm

print(f'Using {best_name} (AUC-ROC: {best_result["AUC-ROC"]:.4f}) for loss estimation.\n')

lgd_scenarios = {
    'Optimistic (30%)': 0.30,
    'Basel Standard (45%)': 0.45,
    'Stressed (60%)': 0.60
}

loss_results = []
for scenario, lgd in lgd_scenarios.items():
    el = pd_predictions * lgd * EAD
    loss_results.append({
        'Scenario': scenario, 'LGD': f'{lgd:.0%}',
        'Portfolio EL ($B)': f'{el.sum()/1e9:,.3f}',
        'Avg EL/Firm ($K)': f'{el.mean()/1e3:,.1f}',
        'Max EL/Firm ($M)': f'{el.max()/1e6:,.2f}'
    })

print('Expected Loss — Sensitivity Analysis:')
pd.DataFrame(loss_results)


### 4.2 Loss Distribution


In [ ]:
LGD = 0.45
el_per_firm = pd_predictions * LGD * EAD

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

axes[0].hist(el_per_firm / 1e6, bins=80, color=ACCENT, edgecolor='white', alpha=0.8)
axes[0].axvline(el_per_firm.mean() / 1e6, color=COLORS[1], ls='--', lw=2,
                label=f'Mean: ${el_per_firm.mean()/1e6:,.2f}M')
axes[0].axvline(np.percentile(el_per_firm, 99) / 1e6, color=COLORS[3], ls='--', lw=2,
                label=f'99th pctile: ${np.percentile(el_per_firm, 99)/1e6:,.2f}M')
axes[0].set_xlabel('Expected Loss per Firm ($M)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Expected Loss Distribution (LGD=45%)', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)

els = [pd_predictions.sum() * lgd * EAD / 1e9 for lgd in lgd_scenarios.values()]
bars = axes[1].bar(list(lgd_scenarios.keys()), els,
                   color=[COLORS[2], ACCENT, COLORS[1]], edgecolor='white')
for bar, el in zip(bars, els):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'${el:,.3f}B', ha='center', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Portfolio Expected Loss ($B)', fontsize=12)
axes[1].set_title('LGD Sensitivity Analysis', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_08_loss_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


### 4.3 Backtest Against Actual Defaults


In [ ]:
test_data = df[df['split'] == 'test'][['fyear', 'target']].copy()
test_data['pd_predicted'] = pd_predictions

backtest = test_data.groupby('fyear').agg(
    actual_rate=('target', 'mean'),
    predicted_rate=('pd_predicted', 'mean'),
    n_firms=('target', 'count'),
    actual_defaults=('target', 'sum')
).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(backtest))
width = 0.35
ax.bar(x - width/2, backtest['actual_rate'] * 100, width,
       label='Actual Default Rate', color=COLORS[1], edgecolor='white')
ax.bar(x + width/2, backtest['predicted_rate'] * 100, width,
       label='Predicted Avg PD', color=ACCENT, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(backtest['fyear'].astype(int))
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Rate (%)', fontsize=12)
ax.set_title('Backtest: Predicted vs. Actual Default Rates (Test Period)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('fig_09_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nBacktest Summary:')
backtest.round(4)


---
## 5. Model Interpretability — Feature Importance


In [ ]:
best_model = results[best_name]['model']

if hasattr(best_model, 'feature_importances_'):
    feat_imp = pd.Series(best_model.feature_importances_, index=FEAT).sort_values(ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    feat_imp.plot(kind='barh', color=ACCENT, ax=ax)
    ax.set_xlabel('Feature Importance', fontsize=12)
    ax.set_title(f'Feature Importance — {best_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig_10_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'Top 10 features ({best_name}):')
    for feat, imp in feat_imp.sort_values(ascending=False).head(10).items():
        print(f'  {feat}: {imp:.4f}')


---
## 6. Calibration & Fairness Audit


In [ ]:
test_analysis = df[df['split'] == 'test'][['target', 'industry']].copy()
test_analysis['pd_predicted'] = pd_predictions
test_analysis['predicted_class'] = best_result['pred']
test_analysis['pd_decile'] = pd.qcut(pd_predictions, 10, labels=False, duplicates='drop')

# Calibration by decile
decile = test_analysis.groupby('pd_decile').agg(
    n=('target', 'count'),
    actual=('target', 'mean'),
    predicted=('pd_predicted', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

# Calibration plot
axes[0].plot([0, decile['predicted'].max()], [0, decile['predicted'].max()],
             'k--', alpha=0.5, label='Perfect Calibration')
axes[0].scatter(decile['predicted'], decile['actual'], s=120, c=ACCENT, zorder=5)
axes[0].set_xlabel('Mean Predicted PD', fontsize=12)
axes[0].set_ylabel('Actual Default Rate', fontsize=12)
axes[0].set_title('Model Calibration by PD Decile', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)

# Sector fairness
ind_fair = test_analysis.groupby('industry').agg(
    actual=('target', 'mean'), predicted=('pd_predicted', 'mean'), n=('target', 'count')
).reset_index()
ind_fair = ind_fair[ind_fair['n'] >= 30].sort_values('actual')

x = np.arange(len(ind_fair))
w = 0.35
axes[1].barh(x - w/2, ind_fair['actual'] * 100, w, label='Actual', color=COLORS[1])
axes[1].barh(x + w/2, ind_fair['predicted'] * 100, w, label='Predicted', color=ACCENT)
axes[1].set_yticks(x)
axes[1].set_yticklabels(ind_fair['industry'], fontsize=9)
axes[1].set_xlabel('Rate (%)', fontsize=12)
axes[1].set_title('Sector Fairness: Actual vs. Predicted Default Rate',
                  fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig('fig_11_calibration_fairness.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSector Fairness (Test Set):')
for _, row in ind_fair.iterrows():
    diff = (row['predicted'] - row['actual']) * 100
    print(f'  {row["industry"]:<22} Actual:{row["actual"]*100:>7.2f}%  Predicted:{row["predicted"]*100:>7.2f}%  Diff:{diff:>+7.2f}%')


---
## 7. Ethical Reflection — Faith-Based Stewardship Perspective

Machine learning models for credit risk carry significant real-world consequences. The predictions generated here influence capital allocation, lending decisions, and ultimately which companies receive financial support.

**Key ethical considerations:**

1. **Transparency & Explainability:** Feature importance analysis reveals which financial ratios drive default risk — not a black box. X11, X12, and X15 emerged as the strongest predictors, providing clear audit trails for model decisions.

2. **Sector Fairness:** Mining and Retail show drastically higher failure rates than other industries. If the model systematically over-predicts default for already-struggling sectors, it could create a self-fulfilling prophecy — restricting capital access precisely when these industries need it most.

3. **Human Oversight:** ML models should inform — not replace — human judgment in credit decisions. The Expected Loss framework provides a quantitative anchor, but experienced analysts must contextualize model outputs within broader economic narratives.

4. **Stewardship of Predictive Power:** From a faith-based perspective, the ability to predict financial distress carries a stewardship responsibility. Predictive models should serve the common good — identifying firms at risk early so that interventions (restructuring, support) can occur, rather than simply enabling risk avoidance that leaves vulnerable entities without recourse.

5. **Data Limitations & Humility:** This model relies on accounting data alone (18 anonymized features). Real-world default is influenced by management quality, macroeconomic shocks, geopolitical events, and factors not captured here. The declining default rate from training (7.9%) to test (2.3%) demonstrates how regime changes can challenge model assumptions. Epistemic humility is essential.


---
## 8. Key Findings & Summary


In [ ]:
print('=' * 70)
print('CORPORATE DEFAULT LOSS ANALYSIS — KEY FINDINGS')
print('=' * 70)
print(f'\nDataset: {len(df):,} firm-year observations ({df["company_name"].nunique():,} companies, 1999-2018)')
print(f'\nBest Model: {best_name}')
print(f'  AUC-ROC:   {best_result["AUC-ROC"]:.4f}')
print(f'  AUC-PR:    {best_result["AUC-PR"]:.4f}')
print(f'  F1-Score:  {best_result["F1"]:.4f}')
print(f'  Threshold: {best_result["thresh"]:.3f}')
portfolio_el = (pd_predictions * 0.45 * EAD).sum()
print(f'\nExpected Loss (Basel Standard LGD=45%, EAD=$100M/firm):')
print(f'  Portfolio EL: ${portfolio_el/1e9:,.3f}B')
print(f'  Average EL per Firm: ${(pd_predictions * 0.45 * EAD).mean()/1e3:,.1f}K')
print(f'\nTop Predictive Features:')
if hasattr(best_model, 'feature_importances_'):
    for i, (feat, imp) in enumerate(feat_imp.sort_values(ascending=False).head(5).items()):
        print(f'  {i+1}. {feat} ({imp:.4f})')
print(f'\nAll figures saved as fig_*.png')
print('=' * 70)


---
## References

- Lombardo, G., Pellegrino, M., Adosoglou, G., Cagnoni, S., Pardalos, P. M., & Poggi, A. (2022). Machine Learning for Bankruptcy Prediction in the American Stock Market: Dataset and Benchmarks. *Future Internet*, 14(8), 244.
- Altman, E. I. (1968). Financial Ratios, Discriminant Analysis and the Prediction of Corporate Bankruptcy. *The Journal of Finance*, 23(4), 589–609.
- Basel Committee on Banking Supervision. (2005). International Convergence of Capital Measurement and Capital Standards: A Revised Framework (Basel II).
- Lundberg, S. M., & Lee, S. I. (2017). A Unified Approach to Interpreting Model Predictions. *Advances in Neural Information Processing Systems*, 30.
- Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002). SMOTE: Synthetic Minority Over-sampling Technique. *Journal of Artificial Intelligence Research*, 16, 321–357.
